In [1]:
# ======================================================
# 1. AMBIENTE E INFRAESTRUTURA
# ======================================================
import os
import json
import sqlite3
from datetime import datetime
from typing import Optional, List, Dict, Any, Literal
from functools import partial, singledispatch

from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import StateGraph, END

# IMPORTS CRUCIAIS PARA O THOTH FUNCIONAR
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser, StrOutputParser

# Configuração do LLM (LM Studio)
llm_thoth = ChatOpenAI(
    model="meta-llama-3.1-8b-instruct",
    base_url="http://localhost:1234/v1",
    api_key="lm-studio",
    temperature=0,
)

# Memória de Checkpoint
db_conn = sqlite3.connect("noosphera_memory.db", check_same_thread=False)
memory = SqliteSaver(conn=db_conn)

print("[✅] Infraestrutura Noosphera pronta.")

[✅] Infraestrutura Noosphera pronta.


In [2]:
# ======================================================
# 2. CONTRATOS DE DOMÍNIO (DDD)
# ======================================================

class ImageMetrics(BaseModel):
    """Value Object: Métricas de qualidade (Baseado no analysis.yaml)"""
    sharpness: float
    contrast: float
    is_noisy: bool = False

class ProcessingStrategy(BaseModel):
    """Entidade: A 'Receita' de processamento decidida pelo Thoth"""
    name: str = Field(..., description="Nome da estratégia (high_accuracy ou fast_scan)")
    engine_mode: int = Field(..., description="OEM Tesseract (1 ou 3)")
    preprocessing_chain: List[str] = Field(..., description="Lista de filtros a aplicar")
    confidence_threshold: float = Field(..., description="Threshold de aceitação")
    reasoning: Optional[str] = Field(None, description="Justificativa do Thoth")

class NoospheraState(BaseModel):
    """Agregado: Estado soberano do sistema"""
    image_path: str
    metrics: Optional[ImageMetrics] = None
    strategy: Optional[ProcessingStrategy] = None
    ocr_result: Optional[str] = None
    confidence: float = 0.0
    attempts: int = 0

print("[✅] Contratos de domínio definidos.")

[✅] Contratos de domínio definidos.


In [3]:
# ======================================================
# 3. GLYPHAR: O MOTOR TÉCNICO (Uso de Partials)
# ======================================================
# Responsabilidade: Execução bruta e configuração de ferramentas

# --- GLYPHAR: MOTOR TÉCNICO ---
def glyphar_ocr_engine(image_path: str, config: Dict[str, Any]) -> str:
    mode = config.get("oem", 1)
    chain = config.get("chain", ["grayscale"])
    return f"[OCR-OEM-{mode}] Processando {image_path} com filtros {chain}."

run_fast_scan = partial(glyphar_ocr_engine, config={"oem": 1, "chain": ["grayscale"]})
run_high_accuracy = partial(glyphar_ocr_engine, config={"oem": 3, "chain": ["grayscale", "denoise", "deskew"]})

# --- THOTH: POLÍTICA FALLBACK (SINGLEDISPATCH) ---
@singledispatch
def thoth_fallback_policy(metrics: Any) -> ProcessingStrategy:
    return ProcessingStrategy(name="fast_scan", engine_mode=1, preprocessing_chain=["grayscale"], confidence_threshold=0.7)

@thoth_fallback_policy.register
def _(metrics: ImageMetrics) -> ProcessingStrategy:
    if metrics.sharpness < 50 or metrics.contrast < 0.3:
        return ProcessingStrategy(name="high_accuracy", engine_mode=3, preprocessing_chain=["grayscale", "denoise"], confidence_threshold=0.9)
    return ProcessingStrategy(name="fast_scan", engine_mode=1, preprocessing_chain=["grayscale"], confidence_threshold=0.8)

In [4]:
# ======================================================
# 4. THOTH: A INTELIGÊNCIA (Uso de Singledispatch)
# ======================================================
# Responsabilidade: Analisar métricas e decidir a política de extração

def analysis_node(state: NoospheraState) -> dict:
    print(f"--- Analisando: {state.image_path} ---")
    # Simulação de métricas conforme seu analysis.yaml
    metrics = ImageMetrics(sharpness=42.0, contrast=0.22) 
    return {"metrics": metrics, "attempts": state.attempts + 1}

def thoth_decision_node(state: NoospheraState) -> dict:
    """O Nó onde o Thoth realmente usa o LLM Studio"""
    parser = JsonOutputParser(pydantic_object=ProcessingStrategy)
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", """Você é o Agente Thoth. Analise as métricas e decida a estratégia.
        Métricas Relevantes (analysis.yaml):
        - Sharpness < 50 ou Contrast < 0.2: 'high_accuracy'.
        - Caso contrário: 'fast_scan'.
        
        {format_instructions}"""),
        ("human", "Métricas: {metrics}\nArquivo: {path}")
    ]).partial(format_instructions=parser.get_format_instructions())

    # Cadeia de decisão
    chain = prompt | llm_thoth | parser
    
    try:
        print("[Thoth] Consultando LLM Studio...")
        strategy = chain.invoke({"metrics": state.metrics.model_dump(), "path": state.image_path})
        # Garante que o retorno é um objeto ProcessingStrategy
        strategy_obj = ProcessingStrategy(**strategy)
    except Exception as e:
        print(f"[!] Erro no LLM, usando Fallback Python: {e}")
        strategy_obj = thoth_fallback_policy(state.metrics)

    print(f"[Thoth] Decisão: {strategy_obj.name}")
    return {"strategy": strategy_obj}

def execution_node(state: NoospheraState) -> dict:
    strat_name = state.strategy.name
    result = run_high_accuracy(state.image_path) if strat_name == "high_accuracy" else run_fast_scan(state.image_path)
    confidence = 0.95 if strat_name == "high_accuracy" else 0.75
    return {"ocr_result": result, "confidence": confidence}

In [5]:
# ======================================================
# 5. NÓS DO GRAFO (Casos de Uso)
# ======================================================

def router_evolutionary(state: NoospheraState) -> Literal["thoth_decisor", "END"]:
    if state.confidence >= state.strategy.confidence_threshold or state.attempts >= 3:
        return "END"
    print(f"[!] Re-avaliando ciclo...")
    return "thoth_decisor"

def build_noosphera():
    builder = StateGraph(NoospheraState)
    builder.add_node("analisador", analysis_node)
    builder.add_node("thoth_decisor", thoth_decision_node)
    builder.add_node("glyphar_executor", execution_node)

    builder.set_entry_point("analisador")
    builder.add_edge("analisador", "thoth_decisor")
    builder.add_edge("thoth_decisor", "glyphar_executor")
    builder.add_conditional_edges("glyphar_executor", router_evolutionary, {"thoth_decisor": "thoth_decisor", "END": END})

    return builder.compile(checkpointer=memory)

# TESTE FINAL
noosphera_graph = build_noosphera()
config = {"configurable": {"thread_id": "session_001"}}
final_output = noosphera_graph.invoke({"image_path": "lacan_doc.jpg"}, config)

print("\n--- RESULTADO FINAL ---")
print(f"Estratégia: {final_output['strategy'].name}")
print(f"Confiança: {final_output['confidence']}")

--- Analisando: lacan_doc.jpg ---
[Thoth] Consultando LLM Studio...
[!] Erro no LLM, usando Fallback Python: 1 validation error for ProcessingStrategy
confidence_threshold
  Input should be a valid number [type=float_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.12/v/float_type
[Thoth] Decisão: high_accuracy

--- RESULTADO FINAL ---
Estratégia: high_accuracy
Confiança: 0.95
